# 第一课 · 手撕 LoRA 微调 Qwen2.5-0.5B

**环境**：Kaggle Notebook（Accelerator = GPU T4 x2，只用其中 1 张即可）
**时长**：课堂 90 min（Part A 30min 实现 · Part B 30min 挂接 · Part C 30min Ablation）
**目标**：
1. 用 **~30 行 PyTorch** 手写 `LoRALinear`，不依赖 `peft`
2. 把 LoRA 注入 Qwen2.5-0.5B 的 `q_proj` / `v_proj`
3. 在小样本中文情感二分类（ChnSentiCorp）上训练并评测
4. 做两个 ablation：**Rank**（r ∈ {1,4,8,32}） + **Target**（QV / QKVO / FFN / 全部）
5. 对训练后的 $BA$ 做 **SVD 可视化**，验证"低秩"假设

> ⚠️ **GQA 预警**：Qwen2.5-0.5B 使用 **GQA（Grouped Query Attention）**，K/V 的维度比 Q 小。后面 ablation 时你会发现 QKVO 的可训练参数**不是** QV 的 2 倍，Section 3 会给出解释。


> ### ⚠️ Kaggle 首次使用必读（课前完成）
>
> 1. **手机号验证**：头像 → Settings → Phone Verification（支持 +86）
> 2. **开启 GPU**：Settings → Accelerator → **GPU T4 x2**
> 3. **开启网络**：Settings → Internet → **On**
>
> 🔴 如果 GPU/Internet 选项是灰色的，说明手机号验证还没完成。不验证 = 无法使用 GPU、无法联网、无法下载模型。


---

## 1. 环境准备与依赖安装

Kaggle 默认已经装好 `torch` + `transformers`，一般只需补装 `datasets` 和 `accelerate`（如果版本太旧）。**不用装 `peft`——第一课坚持手写。**


In [ ]:
# 检查 Python / PyTorch / GPU 环境（自动适配 Kaggle CUDA / Mac MPS / CPU）
import sys, torch, platform
print(f"Python     : {sys.version.split()[0]}")
print(f"PyTorch    : {torch.__version__}")
print(f"Platform   : {platform.platform()}")
print(f"CUDA       : {torch.cuda.is_available()} | Device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU        : {torch.cuda.get_device_name(0)}")
    print(f"Memory     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    print(f"MPS        : True  (Apple Silicon GPU)")
else:
    print(f"⚠️  仅 CPU 模式，训练会非常慢（建议换到 Kaggle GPU）")


In [ ]:
# 设置随机种子 + 全局常量，保证可复现
import random, numpy as np, torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 自动选设备：Kaggle = cuda，Mac M1/M2/M3/M4 = mps，兜底 cpu
# 整份 notebook 都用 DEVICE 这个变量，不用再区分环境
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

MODEL_NAME = "Qwen/Qwen2.5-0.5B"    # 也可换成 Qwen2.5-0.5B-Instruct
MAX_LEN = 128
print(f"Seed={SEED}  Device={DEVICE}  Model={MODEL_NAME}")


---

## 2. 加载 Qwen2.5-0.5B 模型与 Tokenizer

Qwen2.5-0.5B 约 **494M 参数**。我们先加载 base 模型，在其基础上**冻结全部权重**再挂 LoRA。

> 💡 网络慢可以用 Kaggle Dataset 离线模型镜像（课前会发）；或者配 `HF_ENDPOINT=https://hf-mirror.com`。


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,   # 教学目的：用 fp32，观察梯度更直观
    trust_remote_code=True,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"模型参数总量：{total_params:,}  (~{total_params/1e6:.1f} M)")
print(f"模型类型    ：{type(model).__name__}")


---

## 3. 探索模型结构：GQA 维度差异科普 ⚠️

**重要概念**：Qwen2.5-0.5B 使用 **GQA（Grouped Query Attention）**——Q 的头数多，但多个 Q 头**共享**一组 K/V 头。这意味着：

- `q_proj.weight.shape` = `[num_attention_heads × head_dim, hidden]`
- `k_proj.weight.shape` = `[num_key_value_heads × head_dim, hidden]`  ← **K/V 维度更小！**
- `v_proj.weight.shape` = 同上
- `o_proj.weight.shape` = `[hidden, num_attention_heads × head_dim]`

所以后面 ablation 对比时会看到：**QKVO 全挂 ≠ QV 的 2 倍**，因为 K/V 本身更小。

下面我们打印第 0 层的各 proj 的 shape，亲眼看看这个差异：


In [ ]:
# 打印模型配置中的 GQA 关键字段
cfg = model.config
print("="*60)
print(f"hidden_size          : {cfg.hidden_size}")
print(f"num_hidden_layers    : {cfg.num_hidden_layers}")
print(f"num_attention_heads  : {cfg.num_attention_heads}     (Q 的头数)")
print(f"num_key_value_heads  : {cfg.num_key_value_heads}     (K/V 的头数 ← GQA!)")
print(f"head_dim             : {cfg.hidden_size // cfg.num_attention_heads}")
print("="*60)

# 抓第 0 层的 attention 模块，打印各 proj 的 shape
layer0_attn = model.model.layers[0].self_attn
print("\n第 0 层 Attention 各投影层的 weight.shape：")
print("-"*60)
for name in ["q_proj", "k_proj", "v_proj", "o_proj"]:
    w = getattr(layer0_attn, name).weight
    print(f"  {name:8s} : {tuple(w.shape)}  → 参数量 = {w.numel():>10,}")
print("-"*60)
print("📌 看到了吗？k_proj / v_proj 的第 0 维比 q_proj 小很多——这就是 GQA。")
print("   所以 'QKVO 全挂' 的 LoRA 参数 ≠ 'QV' 的 2 倍，而是略多一点。")


---

## 4. 实现 LoRA 模块（手撕版）

**数学公式**（LoRA, Hu et al. 2021）：

$$h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r} \cdot B A x$$

其中：
- $W_0 \in \mathbb{R}^{d_\text{out} \times d_\text{in}}$：原始权重，**冻结**
- $A \in \mathbb{R}^{r \times d_\text{in}}$：高斯初始化 $\mathcal{N}(0, \sigma^2)$
- $B \in \mathbb{R}^{d_\text{out} \times r}$：**零初始化**
- $\alpha$：缩放超参，通常 $\alpha = 2r$ 或 $\alpha = 16$
- $r \ll \min(d_\text{in}, d_\text{out})$：低秩维度

**关键点**：
1. $B=0$ 初始化 → 训练起点 $\Delta W = 0$，等价于原模型，不会破坏已学知识
2. 只训练 $A, B$，参数量 = $r(d_\text{in} + d_\text{out})$，远小于 $d_\text{in} \cdot d_\text{out}$
3. 推理时可以 merge：$W = W_0 + \frac{\alpha}{r} BA$，**零额外开销**


In [ ]:
import torch.nn as nn
import math


class LoRALinear(nn.Module):
    """
    给一个 nn.Linear 外挂 LoRA：h = W0 x + (alpha/r) * B A x
    W0 冻结，只训练 A 和 B。
    """
    def __init__(self, base_linear: nn.Linear, r: int = 8, alpha: int = 16):
        super().__init__()
        self.base = base_linear
        # 冻结原始权重
        for p in self.base.parameters():
            p.requires_grad = False

        in_features  = base_linear.in_features
        out_features = base_linear.out_features

        # A: 高斯初始化（kaiming-like），B: 零初始化
        self.lora_A = nn.Parameter(torch.empty(r, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

    def forward(self, x):
        # 原始前向（不计梯度）
        base_out = self.base(x)
        # LoRA 分支
        # ───────────────────────────────────────────────────────────────
        # 数学公式：h = W0·x + (α/r) · B·A·x         （列向量 x，右乘）
        # 代码实现：h = base(x) + (α/r) · x · Aᵀ · Bᵀ
        # 为什么反过来写？因为 PyTorch 里 x 是行向量 shape=[batch, seq, d_in]，
        #   对行向量而言 (BA)x 的等价写法是 x·(BA)ᵀ = x·Aᵀ·Bᵀ。
        # 两种写法**完全等价**，只是约定不同。
        # ───────────────────────────────────────────────────────────────
        lora_out = (x @ self.lora_A.T) @ self.lora_B.T
        return base_out + lora_out * self.scaling


# 小 sanity check：B=0 时 LoRA 输出应等于原 Linear
_linear = nn.Linear(16, 8).to(DEVICE)
_lora   = LoRALinear(_linear, r=4, alpha=8).to(DEVICE)
_x = torch.randn(2, 16, device=DEVICE)
diff = (_lora(_x) - _linear(_x)).abs().max().item()
print(f"B=0 时 LoRA 与原 Linear 输出最大差：{diff:.2e}  (应接近 0 ✅)")


---

## 5. 注入 LoRA 到目标层

写一个**通用的 `inject_lora`**：递归遍历模型，找到名字匹配 `target_names` 的 `nn.Linear`，用 `LoRALinear` 替换它。

> 💡 这个函数对 **GQA 完全透明**——它只看 `nn.Linear`，不关心 in/out features 是否对称。所以 Qwen GQA 的 `k_proj` / `v_proj` 会被自动包成 LoRA，无需特殊处理。


In [ ]:
def inject_lora(model: nn.Module, target_names=("q_proj", "v_proj"),
                r: int = 8, alpha: int = 16) -> int:
    """
    遍历 model，把所有 child name 在 target_names 里且是 nn.Linear 的层
    替换成 LoRALinear。先冻结全部参数，再让 LoRA 参数可训练。

    返回替换的层数。
    """
    # 1) 先冻结全部
    for p in model.parameters():
        p.requires_grad = False

    # 2) 遍历替换
    count = 0
    for module in model.modules():
        for child_name, child in list(module.named_children()):
            if isinstance(child, nn.Linear) and child_name in target_names:
                new_layer = LoRALinear(child, r=r, alpha=alpha).to(child.weight.device)
                setattr(module, child_name, new_layer)
                count += 1
    return count


def count_trainable(model: nn.Module):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total, trainable / total * 100


---

## 6. 对比可训练参数量：QV vs QKVO（Ablation 准备）

> ⚠️ **重要观察**：因为 GQA，QKVO 的 LoRA 参数**不是** QV 的 2 倍。下面用一个独立函数**干算参数量**（不真的注入），方便对比多种配置。


In [ ]:
def estimate_lora_params(model: nn.Module, target_names, r: int = 8) -> int:
    """不真的注入，仅估算：对每个匹配层，LoRA 参数量 = r*(in + out)"""
    total = 0
    for module in model.modules():
        for child_name, child in module.named_children():
            if isinstance(child, nn.Linear) and child_name in target_names:
                total += r * (child.in_features + child.out_features)
    return total


configs = {
    "Q only"      : ("q_proj",),
    "V only"      : ("v_proj",),
    "QV"          : ("q_proj", "v_proj"),
    "QKV"         : ("q_proj", "k_proj", "v_proj"),
    "QKVO"        : ("q_proj", "k_proj", "v_proj", "o_proj"),
    "FFN (gate+up+down)": ("gate_proj", "up_proj", "down_proj"),
    "ALL (Attn + FFN)"  : ("q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"),
}

print(f"{'Config':<22} | {'LoRA params (r=8)':>18} | {'% of base':>10}")
print("-" * 58)
base_total = sum(p.numel() for p in model.parameters())
for name, targets in configs.items():
    n = estimate_lora_params(model, targets, r=8)
    pct = n / base_total * 100
    print(f"{name:<22} | {n:>18,} | {pct:>9.3f}%")
print("-" * 58)
print("📌 注意 'QKVO' 不等于 'QV' 的 2 倍——这是 GQA 的痕迹。")


In [ ]:
# 真正注入一次：主线配置 = QV, r=8, alpha=16
from copy import deepcopy

# 保留一份原始模型引用，后面做推理对比要用
base_model = model           # 注入会就地修改，下面会看到效果

num_replaced = inject_lora(model, target_names=("q_proj", "v_proj"),
                           r=8, alpha=16)
trainable, total, pct = count_trainable(model)
print(f"替换层数     ：{num_replaced}")
print(f"可训练参数   ：{trainable:,}")
print(f"总参数       ：{total:,}")
print(f"可训练占比   ：{pct:.3f} %   ← 这就是 LoRA 的魔法 ✨")


---

## 7. 准备训练数据集

**任务**：中文情感二分类（ChnSentiCorp）——输入一段评论，输出"正面" or "负面"。

虽然底座是 CausalLM，我们用**指令微调**的方式做分类：
- Prompt 模板：`评论：{text}\n情感：`
- Target：`正面` / `负面`
- 用 CLM loss，只在 target 部分算 loss（实现上用 `labels` 对 prompt 段填 `-100`）

为了课堂演示快，我们只取 **1000 条训练 + 200 条验证**。


In [ ]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

# ChnSentiCorp 在 HF Hub 的镜像：lansinuote/ChnSentiCorp（label: 0=负面, 1=正面）
raw = load_dataset("lansinuote/ChnSentiCorp")
print(raw)

LABEL2TEXT = {0: "负面", 1: "正面"}

def build_prompt(text: str) -> str:
    return f"评论：{text}\n情感："


class SentimentDataset(Dataset):
    def __init__(self, hf_split, tokenizer, max_len=128, limit=None):
        self.samples = hf_split.select(range(limit)) if limit else hf_split
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ex = self.samples[idx]
        prompt = build_prompt(ex["text"])
        target = LABEL2TEXT[ex["label"]] + self.tok.eos_token

        # 分别编码，拼接，记录 prompt 长度以便 mask
        p_ids = self.tok(prompt, add_special_tokens=False)["input_ids"]
        t_ids = self.tok(target, add_special_tokens=False)["input_ids"]
        input_ids = (p_ids + t_ids)[: self.max_len]
        labels = ([-100] * len(p_ids) + t_ids)[: self.max_len]

        # pad 到 max_len
        pad = self.max_len - len(input_ids)
        input_ids = input_ids + [self.tok.pad_token_id] * pad
        labels    = labels    + [-100] * pad
        attn      = [1] * (self.max_len - pad) + [0] * pad

        return {
            "input_ids":      torch.tensor(input_ids),
            "attention_mask": torch.tensor(attn),
            "labels":         torch.tensor(labels),
        }


train_ds = SentimentDataset(raw["train"],      tokenizer, MAX_LEN, limit=1000)
val_ds   = SentimentDataset(raw["validation"], tokenizer, MAX_LEN, limit=200)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False)
print(f"train = {len(train_ds)}  val = {len(val_ds)}")

# 瞄一眼样例
sample = train_ds[0]
print("\n样例 input_ids 前 30 token:", tokenizer.decode(sample["input_ids"][:30]))


---

## 8. 配置优化器与训练循环

**关键**：只把 `requires_grad=True` 的参数（即 LoRA 的 A、B）传给 AdamW，其余 499M 参数不进优化器 → 显存和速度都省很多。


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR


def make_optimizer_and_scheduler(model, lr=1e-3, total_steps=200):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optim = AdamW(trainable_params, lr=lr)
    sched = CosineAnnealingLR(optim, T_max=total_steps)
    return optim, sched


@torch.no_grad()
def eval_loss(model, loader):
    model.eval()
    total_loss, n = 0.0, 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(**batch)
        total_loss += out.loss.item() * batch["input_ids"].size(0)
        n += batch["input_ids"].size(0)
    model.train()
    return total_loss / n


def train_loop(model, train_loader, val_loader,
               epochs=2, lr=1e-3, log_every=20):
    """返回 history: list of dicts"""
    total_steps = epochs * len(train_loader)
    optim, sched = make_optimizer_and_scheduler(model, lr=lr,
                                                 total_steps=total_steps)
    history = []
    step = 0
    model.train()
    for ep in range(epochs):
        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss
            optim.zero_grad()
            loss.backward()
            optim.step()
            sched.step()
            step += 1
            if step % log_every == 0 or step == total_steps:
                history.append({"step": step, "train_loss": loss.item(),
                                "lr": sched.get_last_lr()[0]})
                print(f"[ep {ep+1}] step {step:>4d}/{total_steps}  "
                      f"loss={loss.item():.4f}  lr={sched.get_last_lr()[0]:.2e}")
    val = eval_loss(model, val_loader)
    print(f"→ val_loss = {val:.4f}")
    history.append({"step": step, "val_loss": val})
    return history


---

## 9. 执行微调训练（主线：QV, r=8, alpha=16）

训练 2 epochs（约 250 step），大概 3–5 分钟。观察 loss 下降曲线。


In [ ]:
import matplotlib.pyplot as plt

history_main = train_loop(model, train_loader, val_loader, epochs=2, lr=1e-3)

# 画 loss 曲线（matplotlib 默认不带中文字体，标签用英文避免豆腐块）
steps  = [h["step"] for h in history_main if "train_loss" in h]
losses = [h["train_loss"] for h in history_main if "train_loss" in h]
plt.figure(figsize=(7, 4))
plt.plot(steps, losses, marker="o")
plt.xlabel("step"); plt.ylabel("train loss")
plt.title("LoRA Training Curve (QV, r=8, alpha=16)")
plt.grid(True, alpha=0.3)
plt.show()


---

## 10. 保存与加载 LoRA 权重

LoRA 最大的亮点之一：adapter **只需要保存 A/B 矩阵**，文件非常小（几 MB 甚至几百 KB），可以独立分发。

下面写两个小工具：`save_lora_state` 和 `load_lora_state`。


In [ ]:
import os

def save_lora_state(model, path):
    """只保存 lora_A 和 lora_B 的 state dict"""
    # 自动创建父目录，避免 Kaggle 之外的环境因 /kaggle/working 不存在而报错
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    lora_state = {
        name: p.detach().cpu()
        for name, p in model.named_parameters()
        if "lora_A" in name or "lora_B" in name
    }
    torch.save(lora_state, path)
    size_mb = os.path.getsize(path) / 1e6
    print(f"已保存 {len(lora_state)} 个 LoRA 张量 → {path}  ({size_mb:.2f} MB)")


def load_lora_state(model, path):
    """把 LoRA 权重加载回已经注入过 LoRA 的 model"""
    lora_state = torch.load(path, map_location=DEVICE)
    own = dict(model.named_parameters())
    missing = []
    for name, tensor in lora_state.items():
        if name in own:
            own[name].data.copy_(tensor.to(DEVICE))
        else:
            missing.append(name)
    print(f"加载完成，{len(lora_state) - len(missing)} 个张量写入；缺失 {len(missing)}")


# 自动选保存路径：Kaggle 用官方工作目录，其他环境（Mac / 本地）放到 notebook 同级的 checkpoints/
SAVE_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./checkpoints"
SAVE_PATH = os.path.join(SAVE_DIR, "lora_qv_r8.pt")
save_lora_state(model, SAVE_PATH)


---

## 11. 推理对比：基座 vs LoRA 微调后

拿几条测试评论，对比**基座模型 vs LoRA 微调后**的预测。

**技巧**：我们已经改过 `model`（LoRA 已注入），要看"基座"的表现，只需把 LoRA 的 B 矩阵临时置零（等价于关闭 LoRA），预测完再恢复。


In [ ]:
from contextlib import contextmanager

@contextmanager
def disable_lora(model):
    """临时把所有 LoRALinear 的 B 置零 → 等价于关闭 LoRA"""
    snapshots = []
    for m in model.modules():
        if isinstance(m, LoRALinear):
            snapshots.append((m, m.lora_B.data.clone()))
            m.lora_B.data.zero_()
    try:
        yield
    finally:
        for m, snap in snapshots:
            m.lora_B.data.copy_(snap)


@torch.no_grad()
def predict_sentiment(model, text, max_new_tokens=4):
    prompt = build_prompt(text)
    ids = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    out = model.generate(**ids, max_new_tokens=max_new_tokens,
                         do_sample=False, pad_token_id=tokenizer.pad_token_id)
    # 只取新生成的 token
    new_tokens = out[0, ids["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


test_samples = [
    "这家酒店服务很好，房间干净，下次还会来",            # 正面
    "房间又小又脏，前台态度也差，完全不推荐",            # 负面
    "性价比很高，位置方便，推荐给大家",                    # 正面
    "床太硬了睡不好，早餐也难吃，不会再来了",            # 负面
]

print(f"{'评论':<40} | {'基座':<8} | {'LoRA 后':<8}")
print("-" * 66)
for text in test_samples:
    with disable_lora(model):
        base_pred = predict_sentiment(model, text)
    lora_pred = predict_sentiment(model, text)
    print(f"{text[:36]:<40} | {base_pred:<8} | {lora_pred:<8}")


---

## 12. Ablation 实验（作业骨架）

下面两个 ablation 是作业的**必做项**，模板已经写好，你只需要改配置跑。

**Ablation A · Rank**：`r ∈ {1, 4, 8, 32}`，固定 target = QV

**Ablation B · Target**：`QV / QKVO / FFN / ALL`，固定 r = 8

> ⏱️ 每个配置训练 2 epochs 约 3–5 分钟。课堂先跑 2 个配置做演示，其余交作业时补齐。

> 💡 每次 ablation 都要**重新加载基座模型**（因为上一轮的 LoRA 已经注入）。用下面的 helper。


In [ ]:
import gc

def fresh_model():
    """重新加载一份干净的 base 模型，用于 ablation"""
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float32, trust_remote_code=True
    ).to(DEVICE)
    return m


def run_ablation(target_names, r, alpha=16, epochs=2, lr=1e-3, tag=""):
    print(f"\n===== [{tag}] target={target_names} r={r} =====")
    m = fresh_model()
    n = inject_lora(m, target_names=target_names, r=r, alpha=alpha)
    tr, tot, pct = count_trainable(m)
    print(f"替换 {n} 层 | 可训练 {tr:,} ({pct:.3f}%)")
    hist = train_loop(m, train_loader, val_loader, epochs=epochs, lr=lr)
    final_val = hist[-1]["val_loss"]

    # 🧹 显存清理：Qwen2.5-0.5B fp32 ≈ 2GB，连跑 4 个 ablation 不清会 OOM
    # 如果你想保留某次训练后的模型用于后续分析，请在这里另存 state_dict
    del m
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {"tag": tag, "trainable": tr, "val_loss": final_val}


# ---- Ablation A: Rank （课堂先跑 r=4 和 r=32，作业补 r=1, r=8）----
rank_results = []
for r in [4, 32]:
    rank_results.append(run_ablation(
        ("q_proj", "v_proj"), r=r, tag=f"rank={r}"
    ))

print("\n--- Rank Ablation 汇总 ---")
print(f"{'Config':<12} | {'Trainable':>12} | {'val_loss':>10}")
for res in rank_results:
    print(f"{res['tag']:<12} | {res['trainable']:>12,} | {res['val_loss']:>10.4f}")


In [ ]:
# ---- Ablation B: Target modules （课堂演示 FFN，作业补 QKVO / ALL） ----
target_configs = {
    "FFN": ("gate_proj", "up_proj", "down_proj"),
    # "QKVO": ("q_proj", "k_proj", "v_proj", "o_proj"),
    # "ALL":  ("q_proj", "k_proj", "v_proj", "o_proj",
    #          "gate_proj", "up_proj", "down_proj"),
}

target_results = []
for name, targets in target_configs.items():
    target_results.append(run_ablation(targets, r=8, tag=name))

print("\n--- Target Ablation 汇总（作业请把上面注释掉的两行打开补齐）---")
print(f"{'Config':<8} | {'Trainable':>12} | {'val_loss':>10}")
for res in target_results:
    print(f"{res['tag']:<8} | {res['trainable']:>12,} | {res['val_loss']:>10.4f}")


---

## 13. 可视化洞察：SVD 奇异值谱（验证"低秩"假设）

把训练后的 $\Delta W = BA$ 合并出来，做 SVD：

$$BA = U \Sigma V^\top$$

画出奇异值 $\sigma_i$ 的衰减曲线——如果少数几个奇异值远大于其他，说明权重更新**确实是低秩的**，LoRA 的假设成立。

下面对**主线模型（QV, r=8）**的第 0、5、10 层（3 个代表性位置）画 SVD 谱。


In [ ]:
def collect_lora_deltas(model, proj_name="q_proj"):
    """返回 dict: layer_idx → ΔW = scaling * B @ A"""
    deltas = {}
    for name, m in model.named_modules():
        if isinstance(m, LoRALinear) and name.endswith(f".{proj_name}"):
            # name like "model.layers.3.self_attn.q_proj"
            try:
                layer_idx = int(name.split(".layers.")[1].split(".")[0])
            except Exception:
                continue
            with torch.no_grad():
                delta = m.scaling * (m.lora_B @ m.lora_A)   # [out, in]
            deltas[layer_idx] = delta.detach().cpu().float()
    return deltas


# 用主线 model（QV, r=8 已训练过）
deltas_q = collect_lora_deltas(model, "q_proj")
deltas_v = collect_lora_deltas(model, "v_proj")
print(f"收集到 q_proj ΔW：{len(deltas_q)} 层 | v_proj ΔW：{len(deltas_v)} 层")
print(f"单个 ΔW shape: {next(iter(deltas_q.values())).shape}")


# 对每层做 SVD，画奇异值谱（matplotlib 标签统一用英文，避免中文字体缺失报豆腐块）
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sample_layers = [0, len(deltas_q)//2, len(deltas_q) - 1]

for ax, (proj_name, deltas) in zip(axes, [("q_proj", deltas_q),
                                            ("v_proj", deltas_v)]):
    for li in sample_layers:
        dW = deltas[li]
        s = torch.linalg.svdvals(dW)
        # 归一化，方便跨层对比
        s = s / s[0]
        ax.plot(s.numpy()[:50], marker="o", ms=3, label=f"layer {li}")
    ax.set_yscale("log")
    ax.set_xlabel("Singular value index")
    ax.set_ylabel(r"$\sigma_i / \sigma_0$")
    ax.set_title(f"SVD spectrum of {proj_name} $\\Delta W$ (r=8)")
    ax.axvline(x=8, color="red", linestyle="--", alpha=0.5, label="r=8 cutoff")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📌 观察：前 ~8 个奇异值远大于其他——这就是 LoRA 能 work 的根本原因。")
print("   ΔW 的'有效秩'非常小，用 r=8 捕捉已经足够。")


---

## 🎓 今日小结

1. **LoRA 核心公式**：$W_0 + \frac{\alpha}{r} BA$，冻结 $W_0$，只训练低秩的 $A, B$
2. **可训练参数占比 ≈ 0.16%**（QV, r=8），显存/时间大幅节省
3. **GQA 的坑**：K/V 维度比 Q 小，所以 QKVO 参数量 **≠** QV × 2
4. **低秩假设**被 SVD 谱验证：前几个奇异值主导整个 $\Delta W$
5. **Ablation 是科研基本功**：rank 和 target 的选择都需要实验来回答

## 📋 作业（一周内提交）

> 详细要求见 [`homework.md`](./homework.md)。核心项：
> - [ ] 跑通 r ∈ {1, 4, 8, 32} 的 Rank Ablation，提交对比表 + loss 曲线图
> - [ ] 补齐 QKVO / ALL 的 Target Ablation
> - [ ] SVD 奇异值图（至少 3 层）+ 一段 200 字分析
> - [ ] 理解第二课将使用的**外贸术语 QA 数据集**（见 `lesson2/data/`）

---

> 🦞 下一课：用 `peft` + QLoRA 训练 Qwen2.5-1.5B 做外贸术语问答。
